In [89]:
# -*- coding: utf-8 -*-
from pathlib import Path
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import gcamreader

def convert_to_mt(row):
    val, unit = row['value'], row['Units']
    if unit == 'Tg':
        return val
    elif unit == 'Gg':
        return val * 1e-3
    elif unit == 'MTC':
        return val * (44.009 / 12.011)
    else:
        raise ValueError(f"Unknown unit: {unit}")

# AR5 100‑yr GWP (no climate–carbon feedbacks)
GWP_AR5 = {
    'CO2':      1,      
    'CH4':     28,      
    'CH4_AGR': 28,
    'CH4_AWB': 28,
    'N2O':    265,      
    'N2O_AGR':265,
    'N2O_AWB':265,
    'HFC125': 3170,     
    'HFC134a':1300,     
    'HFC143a':4800,     
    'HFC23': 12400,     
    'HFC32':   677,     
    'HFC43':  1650,     
    'HFC227ea':3350,    
    'HFC236fa':8060,    
    'SF6':   23500,     
    'C2F6':  11100,     
    'CF4':    6630,     
}

In [90]:
# =========================================================
# Config
# =========================================================
PROJECT_PATH   = Path("/data/project/tae/gcam-core")
DB_REL_PATH    = "../output"
DB_FILE        = "database_basexdb_korea_2035_v6"
QUERY_FILE     = Path("..") / "output" / "queries" / "Main_queries.xml"

REGION = "South Korea"
SCENARIOS = [
    "Current-Policies-Med", "High-Ambition-Med",
    "Current-Policies-Low", "High-Ambition-Low",
    "Current-Policies-High","High-Ambition-High"
]

# Query indices
Q_CO2   = 262
Q_NONCO2= 270

# Historical emissions (MtCO2e)
EMISS_2018 = 742.3
INTL_OFFSET = 32.3


# Paths
EXTDATA_XLSX = "./extdata/gir2025.xlsx"

# Output figs
FIG_TOTAL = "./fig/total_emission_sec.png"
FIG_HIST  = "./fig/korea_hist.png"

In [91]:
# =========================================================
# DB helpers
# =========================================================
def connect_db():
    return gcamreader.LocalDBConn(DB_REL_PATH, DB_FILE)

def run_query(conn, q_idx):
    queries = gcamreader.parse_batch_query(os.fspath(QUERY_FILE))
    q = queries[q_idx]
    df = conn.runQuery(q, scenarios=SCENARIOS, regions=[REGION])
    df["scenario"] = df["scenario"].str.split(",").str[0]
    return df

In [92]:
# =========================================================
# Historical loaders
# =========================================================
def read_gir_sheet(sheet):
    return pd.read_excel(
        EXTDATA_XLSX, sheet_name=sheet, skiprows=3,
        index_col=1, skipfooter=9, engine="openpyxl"
    ).transpose().iloc[1:, ]

def load_historical():
    # gas-specific totals
    gas = read_gir_sheet("온실가스")
    # dfCH4 = read_gir_sheet("CH4")
    # dfN2O = read_gir_sheet("N2O")
    # dfSF6 = read_gir_sheet("SF6")
    # dfHFC = read_gir_sheet("HFCs")
    # dfPFC = read_gir_sheet("PFCs")

    # gas = pd.DataFrame({
    #     "CO2": dfCO2["총배출량"],
    #     "CH4": dfCH4["총배출량"],
    #     "N2O": dfN2O["총배출량"],
    #     "F-Gases": dfSF6["총배출량"] + dfHFC["총배출량"] + dfPFC["총배출량"],
    # }).reset_index(names=["Year"])

    # stacked (Gg → Mt)
    gas["CO2_stack"] = gas["CO2"] / 1000
    gas["CH4_stack"] = (gas["CO2"] + gas["CH4"]) / 1000
    gas["N2O_stack"] = (gas["CO2"] + gas["CH4"] + gas["N2O"]) / 1000
    gas["FGas_stack"] = (gas["CO2"] + gas["CH4"] + gas["N2O"] + gas["F-Gases"]) / 1000

    # “Net emissions” timeseries for the scenario legend line
    net = pd.read_excel(
        EXTDATA_XLSX, skiprows=3, index_col=1, skipfooter=9, engine="openpyxl"
    ).transpose().iloc[1:, ].reset_index()
    # net has columns: 'index' (year), '순배출량' (Gg CO2e)
    net.rename(columns={"index": "Year"}, inplace=True)
    net["NetMt"] = net["순배출량"] / 1000
    return gas, net

In [93]:
# “Net emissions” timeseries for the scenario legend line
net_df = pd.read_excel(
    EXTDATA_XLSX, skiprows=3, index_col=1, skipfooter=9, engine="openpyxl"
).transpose().iloc[1:, ].reset_index()
# net has columns: 'index' (year), '순배출량' (Gg CO2e)
net_df.rename(columns={"index": "Year"}, inplace=True)
net_df["NetMt"] = net_df["순배출량"] / 1000
net_df

분야·부문/연도,Year,총배출량,순배출량,에너지,A. 연료연소,1. 에너지산업,a. 공공전기 및 열 생산,b. 석유정제,c. 고체연료 제조 및 기타 에너지 산업,2. 제조업 및 건설업,...,2. 바이오가스시설에서의 혐기성 소화,C. 폐기물소각 및 노천소각,1. 폐기물소각,2. 노천소각,D. 하폐수처리,1. 하수처리,2. 폐수처리,3. 기타,E. 기타,NetMt
0,1990,310578.34,271615.09,234475.75,223481.61,42026.93,37174.66,4193.73,658.54,72169.01,...,0.00,560.85,560.85,0.0,729.52,631.05,98.48,0.0,0.0,271.61509
1,1991,341241.27,306225.60,257328.69,246864.64,49945.73,43773.82,4978.80,1193.10,85306.81,...,0.00,746.36,746.36,0.0,911.36,793.36,118.00,0.0,0.0,306.22560
2,1992,368809.57,334622.72,273752.07,264236.12,59044.99,51282.99,6097.80,1664.21,88857.58,...,0.00,879.08,879.08,0.0,1070.06,919.56,150.50,0.0,0.0,334.62272
3,1993,406802.15,374169.23,303458.93,294738.32,66967.19,58078.04,6787.82,2101.33,97533.10,...,0.00,1083.73,1083.73,0.0,889.59,715.65,173.94,0.0,0.0,374.16923
4,1994,432769.28,397753.22,323158.10,315376.07,81318.26,71719.31,7046.10,2552.86,102910.98,...,0.00,1547.93,1547.93,0.0,1513.83,1277.13,236.71,0.0,0.0,397.75322
5,1995,464497.63,431236.45,347962.44,340964.77,91952.43,79555.15,9291.77,3105.51,104629.69,...,0.00,2212.85,2212.85,0.0,1520.26,1238.24,282.02,0.0,0.0,431.23645
6,1996,501079.14,464394.65,381519.41,374951.00,115575.06,96643.67,10525.32,8406.07,106236.46,...,0.00,1607.51,1607.51,0.0,1561.24,1406.53,154.71,0.0,0.0,464.39465
7,1997,526063.93,484380.56,398735.84,392509.74,124958.96,104573.97,13160.26,7224.74,107331.55,...,0.00,2014.38,2014.38,0.0,1381.70,1231.28,150.42,0.0,0.0,484.38056
8,1998,460219.69,410168.46,340200.65,334224.21,112389.64,92499.17,13398.10,6492.37,97052.38,...,0.00,1901.37,1901.37,0.0,1631.16,1504.14,127.02,0.0,0.0,410.16846
9,1999,500569.62,442315.49,372901.35,367093.36,122425.73,101887.04,13874.42,6664.27,104531.80,...,0.00,3245.30,3245.30,0.0,1728.48,1573.45,155.02,0.0,0.0,442.31549


In [94]:
dfE = pd.read_excel(EXTDATA_XLSX, sheet_name="온실가스", skiprows=3, index_col=1, skipfooter=9, engine="openpyxl").transpose().iloc[1:, ]
list(dfE.columns)

['총배출량',
 '순배출량',
 '에너지',
 'A. 연료연소',
 '1. 에너지산업',
 'a. 공공전기 및 열 생산',
 'b. 석유정제',
 'c. 고체연료 제조 및 기타 에너지 산업',
 '2. 제조업 및 건설업',
 'a. 철강',
 'b. 비철금속',
 'c. 화학',
 'd. 펄프, 제지 및 인쇄',
 'e. 식음료품 가공 및 담배 제조',
 'f. 비금속광물',
 'g.기타',
 '  1. 기계 제조',
 '  2. 수송장비 제조',
 '  3. 광업',
 '  4. 목재 및 목제품',
 '  5. 건설업',
 '  6. 섬유 및 가죽',
 '  7. 기타',
 '3. 수송',
 'a. 국내항공',
 'b. 도로수송',
 'c. 철도',
 'd. 국내해운',
 'e. 기타수송',
 '4. 기타',
 'a. 상업/공공',
 'b. 가정',
 'c. 농업/임업/어업',
 '    농업/임업',
 '    어업',
 '5. 미분류',
 'B. 탈루',
 '1. 고체연료',
 '2.  석유 및 천연가스',
 'a.석유',
 'b.천연가스',
 'c.탈기 및 소각',
 'C.이산화탄소 수송 및 저장',
 '산업공정 및 제품사용',
 'A. 광물산업',
 '1. 시멘트생산',
 '2. 석회생산',
 '3. 유리생산',
 '4. 탄산염의 기타 공정 사용',
 'B. 화학산업',
 '1. 암모니아 생산',
 '2. 질산 생산',
 '3. 아디프산 생산',
 '4. 카프로락탐, 글리옥살, 글리옥실산 생산',
 '5. 카바이드 생산',
 '6. 이산화티타늄 생산',
 '7. 소다회 생산',
 '8. 석유화학제품 및 카본블랙 생산',
 '9. 불소화합물 생산',
 '10. 기타',
 'C. 금속산업',
 '1. 철강생산',
 '2. 합금철 생산',
 '3. 알루미늄 생산',
 '4. 마그네슘 생산',
 '5.  납 생산',
 '6.  아연 생산',
 '7.  기타',
 'D. 비에너지 연료 및 용매 사용',
 '1. 윤활유 사용',
 '2. 파라핀 왁스 사용',


In [95]:
dfE.head()

분야·부문/연도,총배출량,순배출량,에너지,A. 연료연소,1. 에너지산업,a. 공공전기 및 열 생산,b. 석유정제,c. 고체연료 제조 및 기타 에너지 산업,2. 제조업 및 건설업,a. 철강,...,1. 퇴비화,2. 바이오가스시설에서의 혐기성 소화,C. 폐기물소각 및 노천소각,1. 폐기물소각,2. 노천소각,D. 하폐수처리,1. 하수처리,2. 폐수처리,3. 기타,E. 기타
1990,310578.34,271615.09,234475.75,223481.61,42026.93,37174.66,4193.73,658.54,72169.01,27836.42,...,0.00,0.0,560.85,560.85,0.0,729.52,631.05,98.48,0.0,0.0
1991,341241.27,306225.60,257328.69,246864.64,49945.73,43773.82,4978.80,1193.10,85306.81,35194.60,...,0.00,0.0,746.36,746.36,0.0,911.36,793.36,118.00,0.0,0.0
1992,368809.57,334622.72,273752.07,264236.12,59044.99,51282.99,6097.80,1664.21,88857.58,37227.61,...,0.00,0.0,879.08,879.08,0.0,1070.06,919.56,150.50,0.0,0.0
1993,406802.15,374169.23,303458.93,294738.32,66967.19,58078.04,6787.82,2101.33,97533.10,41604.70,...,0.00,0.0,1083.73,1083.73,0.0,889.59,715.65,173.94,0.0,0.0
1994,432769.28,397753.22,323158.10,315376.07,81318.26,71719.31,7046.10,2552.86,102910.98,41368.27,...,4.26,0.0,1547.93,1547.93,0.0,1513.83,1277.13,236.71,0.0,0.0


In [96]:
serConv = dfE['a. 공공전기 및 열 생산'] / 1000
serInd = (dfE['2. 제조업 및 건설업'] + dfE['산업공정 및 제품사용'] + dfE['b. 석유정제'] + dfE['c. 고체연료 제조 및 기타 에너지 산업'] - dfE['F. 오존층파괴물질의 대체물질 사용']) / 1000
serBlds = dfE['4. 기타'] / 1000
serTrn = dfE['3. 수송']  / 1000
serRfg = dfE['F. 오존층파괴물질의 대체물질 사용'] / 1000
serAg = dfE['농업'] / 1000
serWaste = dfE['폐기물'] / 1000
serH2 = 0
serEtc = (dfE['5. 미분류'] + dfE['B. 탈루']) / 1000
serLulucf = dfE['LULUCF'] / 1000

In [97]:
sec = pd.DataFrame({
    "Power": serConv,
    "Industry": serInd,
    "Transportation": serTrn,
    "Buildings": serBlds,
    "Refrigerant": serRfg,
    "Agriculture": serAg,
    "Waste": serWaste,
    "Other": serEtc,
    "LULUCF": serLulucf,
}).reset_index(names=["Year"])

In [98]:
colors = {
    "Power": "#6e6e6e",   # gray
    "Industry": "#4472c4",     # blue
    "Transportation": "#ffc000", # yellow
    "Buildings": "#70ad47",    # green
    "Refrigerant": "#00B5F7", # yellow
    "Agriculture": "#FD3216",  # orange
    "Waste": "#E15F99",        # light gray
    "Other": "#BCBD22",        # brown
    "LULUCF": "#66c2a5"        # greenish for negative
}

In [99]:
# =========================================================
# Model outputs → MtCO2e (AR5)
# =========================================================
def build_model_mtco2e(conn):
    df_co2   = run_query(conn, Q_CO2)
    df_co2["GHG"] = "CO2"

    df_nonco2= run_query(conn, Q_NONCO2)

    df = pd.concat([df_co2, df_nonco2], ignore_index=True)
    df["emiss(MT)"] = df.apply(convert_to_mt, axis=1)  # uses your utils.convert_to_mt
    df["gwpAr5"]    = df["GHG"].map(GWP_AR5).astype(float)
    df["MTCO2eq"]   = df["emiss(MT)"] * df["gwpAr5"]
    return df

In [100]:
# =========================================================
# LULUCF negative-emissions assumptions (MtCO2e, subtracted)
# =========================================================
LULUCF_NEG_EMISSIONS = {
    "Current-Policies-Med":  {2005: 57.5, 2010: 57.3, 2015: 47.8, 2020: 38.8, 2025: 42.1, 2030: 42.1, 2035: 42.1},
    # 42.1 -> average value of recent 10 years (2014-2023), 48.2 -> highest value (2016)
    "High-Ambition-Med": {2005: 57.5, 2010: 57.3, 2015: 47.8, 2020: 38.8, 2025: 42.1, 2030: 45.2, 2035: 48.2},
    "Current-Policies-High": {2005: 57.5, 2010: 57.3, 2015: 47.8, 2020: 38.8, 2025: 42.1, 2030: 42.1, 2035: 42.1},
    "High-Ambition-High":{2005: 57.5, 2010: 57.3, 2015: 47.8, 2020: 38.8, 2025: 42.1, 2030: 45.2, 2035: 48.2},
    "Current-Policies-Low":  {2005: 57.5, 2010: 57.3, 2015: 47.8, 2020: 38.8, 2025: 42.1, 2030: 42.1, 2035: 42.1},
    "High-Ambition-Low": {2005: 57.5, 2010: 57.3, 2015: 47.8, 2020: 38.8, 2025: 42.1, 2030: 45.2, 2035: 48.2},
}

def apply_lulucf_sinks(df_out):
    """
    Apply LULUCF negative-emissions (sinks) to totals.

    Parameters
    ----------
    df_out : DataFrame indexed by ['scenario','Year'] with a 'value' column in MtCO2e
        Represents gross economy-wide emissions (excl. intl. aviation/shipping).

    Notes
    -----
    Values in LULUCF_NEG_EMISSIONS are *MtCO2e sinks* (positive numbers),
    which are subtracted from gross emissions: net = gross - sink.
    """
    for scen, year_map in LULUCF_NEG_EMISSIONS.items():
        for yr, sink_mtco2e in year_map.items():
            key = (scen, yr)
            if key in df_out.index:
                df_out.loc[key, "value"] -= sink_mtco2e

In [101]:
# =========================================================
# Plot builders
# =========================================================
def add_gas_stack(fig, gas_df):
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
                             line=dict(color='rgba(0,0,0,0)'),
                             name='<br><br><br><b>Gas</b>',
                             showlegend=True, hoverinfo='skip'))
    fig.add_trace(go.Scatter(x=gas_df["Year"], y=gas_df["CO2_stack"],
                             mode='lines', name='CO2', fill='tozeroy',
                             line=dict(width=0.5, color='grey')))
    fig.add_trace(go.Scatter(x=gas_df["Year"], y=gas_df["CH4_stack"],
                             mode='lines', name='CH4', fill='tonexty',
                             line=dict(width=0.5, color='green')))
    fig.add_trace(go.Scatter(x=gas_df["Year"], y=gas_df["N2O_stack"],
                             mode='lines', name='N2O', fill='tonexty',
                             line=dict(width=0.5, color='purple')))
    fig.add_trace(go.Scatter(x=gas_df["Year"], y=gas_df["FGas_stack"],
                             mode='lines', name='F-Gases', fill='tonexty',
                             line=dict(width=0.5, color='yellow')))

def add_scenario_lines(fig, df_out, cp_name, ep_name, start_year=2025):
    # separator in legend
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
                             line=dict(color='rgba(0,0,0,0)'),
                             name='<br><br><br><b>Scenario</b>',
                             showlegend=True, hoverinfo='skip'))
    # historical net
    # (added outside with add_hist_net for clarity)

    # scenario lines
    mask_cp = (df_out["scenario"] == cp_name) & (df_out["Year"] >= start_year)
    mask_ep = (df_out["scenario"] == ep_name) & (df_out["Year"] >= start_year)
    fig.add_trace(go.Scatter(
        x=df_out.loc[mask_cp, "Year"], y=df_out.loc[mask_cp, "value"],
        mode='lines', name='Current Policies',
        line=dict(color='#636EFA', width=1.5)
    ))
    fig.add_trace(go.Scatter(
        x=df_out.loc[mask_ep, "Year"], y=df_out.loc[mask_ep, "value"],
        mode='lines', name='High Ambition',
        line=dict(color='#00CC96', width=1.5)
    ))

def add_scenario_bands(fig, df_out, low_scen, high_scen, fillcolor):
    yrs  = df_out.loc[df_out["scenario"] == low_scen, "Year"].values
    low  = df_out.loc[df_out["scenario"] == low_scen, "value"].values
    high = df_out.loc[df_out["scenario"] == high_scen, "value"].values
    if len(yrs) and len(low) and len(high) and len(low) == len(high):
        fig.add_trace(go.Scatter(
            x=list(yrs) + list(yrs[::-1]),
            y=list(high) + list(low[::-1]),
            fill='toself', fillcolor=fillcolor,
            line=dict(color='rgba(255,255,255,0)'),
            hoverinfo="skip", showlegend=False
        ))
        
def style_axes(fig, x_range, y_range, x_ticks, y_ticks, x_title="Year", y_title="Emission (MtCO2e)"):
    fig.update_layout(
        legend_traceorder="normal", legend_title='', legend_font_size=15,
        plot_bgcolor='rgba(0,0,0,0)', width=1200, height=700,
        xaxis=dict(showgrid=False, title=x_title, title_font_size=25,
                   tickvals=x_ticks, tickfont_size=15, range=x_range),
        yaxis=dict(showgrid=False, title=y_title, title_font_size=25,
                   tickvals=y_ticks, tickfont_size=15, range=y_range),
    )

def add_grid(fig, x0, x1, y0, y1, x_step=5, y_step=100):
    for x in range(int(x0), int(x1)+1, x_step):
        fig.add_shape(type="line", x0=x, x1=x, y0=y0, y1=y1,
                      line=dict(color="LightGrey", width=1, dash="dash"), layer='below')
    for y in range(int(y0), int(y1)+1, y_step):
        fig.add_shape(type="line", x0=x0, x1=x1, y0=y, y1=y,
                      line=dict(color="LightGrey", width=1, dash="dash"), layer='below')
    fig.add_shape(type="line", x0=x0, x1=x1, y0=0, y1=0,
                  line=dict(color="LightGrey", width=1, dash="dash"), layer='above')

In [102]:
conn = connect_db()

Database scenarios: Current-Policies-Med, Current-Policies-Med, High-Ambition-Med, High-Ambition-Med, Current-Policies-Med, Current-Policies-Med, High-Ambition-High, Current-Policies-High, High-Ambition-Low, Current-Policies-Low, High-Ambition-Med-AI, Current-Policies-Med-AI, High-Ambition-Med-CPO2040


In [103]:
# ----- Load model outputs → MtCO2e -----
dfGHG = build_model_mtco2e(conn)
# drop international shipping/aviation
dfOut = (
    dfGHG[~dfGHG["sector"].isin(["trn_aviation_intl", "trn_shipping_intl"])]
    .groupby(["scenario", "Year"], as_index=False)["MTCO2eq"].sum()
    .rename(columns={"MTCO2eq": "value"})
)

# LULUCF sinks
dfOut = dfOut.set_index(["scenario", "Year"])
apply_lulucf_sinks(dfOut)
dfOut = dfOut.reset_index()

# Precompute 2035 reductions vs 2018
def pct_red(v): return (1 - (v / EMISS_2018)) * 100
cp_med_2035 = dfOut.set_index(["scenario", "Year"])["value"].get(("Current-Policies-Med", 2035))
ep_med_2035 = dfOut.set_index(["scenario", "Year"])["value"].get(("High-Ambition-Med", 2035))
red_cp = pct_red(cp_med_2035) if cp_med_2035 is not None else np.nan
red_ep = pct_red(ep_med_2035) if ep_med_2035 is not None else np.nan

In [104]:
def add_sector_stack(fig, sec_df):
    """Add sectoral stacked area chart to an existing Plotly figure."""
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='lines',
        line=dict(color='rgba(0,0,0,0)'),
        name='<b>Sector</b>', showlegend=True, hoverinfo='skip'
    ))

    # Sector order (bottom to top)
    sectors = [
        ("Power", "Power", "#6e6e6e"),
        ("Industry", "Industry_stack", "#4472c4"),
        ("Transportation", "Transportation_stack", "rgb(255,217,47)"),
        ("Buildings", "Buildings_stack", "#70ad47"),
        ("Refrigerant", "Refrigerant_stack", "#00B5F7"),
        ("Agriculture", "Agriculture_stack", "#FF7F02"),
        ("Waste", "Waste_stack", "#E15F99"),
        ("Other", "Other_stack", "#BCBD22")
    ]

    for i, (name, stack_col, color) in enumerate(sectors):
        fill_type = 'tozeroy' if i == 0 else 'tonexty'

        fig.add_trace(go.Scatter(
            x=sec_df["Year"], y=sec_df[stack_col],
            mode="lines",
            name=name,
            fill=fill_type,
            line=dict(width=0.5, color=color),
            fillcolor=color,        # ← solid fill, no transparency
            hovertemplate=f"{name}: %{{y:.1f}} MtCO₂e<br>Year: %{{x}}<extra></extra>"
        ))


    # LULUCF (negative area)
    fig.add_trace(go.Scatter(
        x=sec_df["Year"], y=sec_df["LULUCF"],
        mode="lines", name="LULUCF", fill="tozeroy",
        line=dict(width=0.5, color="#2CA02C")
    ))

    return fig

In [105]:
dfS = dfOut[(dfOut['Year']==2035)].copy()
dfS['rr'] = (1 - dfS['value'] / 742.3) * 100
dfS

,scenario,Year,value,rr
8,Current-Policies-High,2035,535.081789,27.915696
17,Current-Policies-Low,2035,447.539609,39.709065
26,Current-Policies-Med,2035,493.399954,33.530924
35,High-Ambition-High,2035,333.491819,55.073175
44,High-Ambition-Low,2035,264.446121,64.374765
53,High-Ambition-Med,2035,301.989807,59.317014


In [106]:
dfOut[(dfOut['Year']==2030)]

,scenario,Year,value
7,Current-Policies-High,2030,601.697954
16,Current-Policies-Low,2030,552.762796
25,Current-Policies-Med,2030,581.708601
34,High-Ambition-High,2030,489.967729
43,High-Ambition-Low,2030,446.432261
52,High-Ambition-Med,2030,474.200460


In [107]:
dfOut[(dfOut['scenario'] == 'Current-Policies-Med')]

,scenario,Year,value
18,Current-Policies-Med,1975,54.590644
19,Current-Policies-Med,1990,295.171616
20,Current-Policies-Med,2005,531.334536
21,Current-Policies-Med,2010,637.680605
22,Current-Policies-Med,2015,702.955896
23,Current-Policies-Med,2020,670.911116
24,Current-Policies-Med,2025,645.020828
25,Current-Policies-Med,2030,581.708601
26,Current-Policies-Med,2035,493.399954


In [108]:
dfOut[(dfOut['scenario'] == 'High-Ambition-Med')]

,scenario,Year,value
45,High-Ambition-Med,1975,54.590644
46,High-Ambition-Med,1990,295.171616
47,High-Ambition-Med,2005,531.334536
48,High-Ambition-Med,2010,637.680605
49,High-Ambition-Med,2015,702.955896
50,High-Ambition-Med,2020,670.911116
51,High-Ambition-Med,2025,645.020828
52,High-Ambition-Med,2030,474.200460
53,High-Ambition-Med,2035,301.989807


In [109]:
1 - 301.989807 / 783.8

0.6147106315386578

In [110]:
(670.911116 - 474.200460) / 10 

19.6710656

In [111]:
742.3 * 0.6

445.37999999999994

In [112]:
(474.200460 - 301.989807) / 5

34.442130600000006

In [113]:
1 - 581.708601 / 742.3

0.21634298666307417

In [114]:
dfHistGas, dfHistNet = sec, net_df

In [115]:
dfHistGas['Industry_stack'] = dfHistGas['Power'] + dfHistGas['Industry']
dfHistGas['Transportation_stack'] = dfHistGas['Industry_stack'] + dfHistGas['Transportation']
dfHistGas['Buildings_stack'] = dfHistGas['Transportation_stack'] + dfHistGas['Buildings']
dfHistGas['Refrigerant_stack'] = dfHistGas['Buildings_stack'] + dfHistGas['Refrigerant']
dfHistGas['Agriculture_stack'] = dfHistGas['Refrigerant_stack'] + dfHistGas['Agriculture']
dfHistGas['Waste_stack'] = dfHistGas['Agriculture_stack'] + dfHistGas['Waste']
dfHistGas['Other_stack'] = dfHistGas['Waste_stack'] + dfHistGas['Other']

In [116]:
dfHistGas.columns

Index(['Year', 'Power', 'Industry', 'Transportation', 'Buildings',
       'Refrigerant', 'Agriculture', 'Waste', 'Other', 'LULUCF',
       'Industry_stack', 'Transportation_stack', 'Buildings_stack',
       'Refrigerant_stack', 'Agriculture_stack', 'Waste_stack', 'Other_stack'],
      dtype='object')

In [117]:
dfHistGas.head()

,Year,Power,Industry,Transportation,Buildings,Refrigerant,Agriculture,Waste,Other,LULUCF,Industry_stack,Transportation_stack,Buildings_stack,Refrigerant_stack,Agriculture_stack,Waste_stack,Other_stack
0,1990,37.17466,114.81904,36.46081,72.64146,0.00000,24.86547,13.43935,11.17753,-38.96325,151.99370,188.45451,261.09597,261.09597,285.96144,299.40079,310.57832
1,1991,43.77382,135.65363,39.66714,67.24069,0.00000,24.91557,14.82209,15.16833,-35.01567,179.42745,219.09459,286.33528,286.33528,311.25085,326.07294,341.24127
2,1992,51.28299,150.08146,45.19005,68.17138,0.00157,25.21480,16.37927,12.48807,-34.18685,201.36445,246.55450,314.72588,314.72745,339.94225,356.32152,368.80959
3,1993,58.07804,166.77962,57.08243,69.97455,0.00429,25.54729,17.43427,11.90168,-32.63292,224.85766,281.94009,351.91464,351.91893,377.46622,394.90049,406.80217
4,1994,71.71931,176.55140,59.19033,69.08881,0.23876,25.76765,19.56331,10.64972,-35.01607,248.27071,307.46104,376.54985,376.78861,402.55626,422.11957,432.76929


In [145]:
# ============================
# Figure 1: Total emissions with bands & annotations
# ============================
fig = go.Figure()
# add_gas_stack(fig, dfHistGas)
add_sector_stack(fig, dfHistGas)



# 2018 reference
fig.add_trace(go.Scatter(
    x=[2018], y=[742.3],
    mode='markers+text',
    marker=dict(color='black', size=5),
    text=f'Baseline: {742.3}',
    textposition='middle right',
    textfont=dict(size=15),
    showlegend=False
))

# # NDC marker (60% of 2018 at 2030)
# fig.add_trace(go.Scatter(
#     x=[2030], y=[783.8 * 0.6],
#     mode='markers+text',
#     marker=dict(color='#750D86', size=7, symbol='triangle-up'),
#     text='2030 NDC', textposition="middle left",
#     textfont=dict(size=15, color="#750D86"),
#     showlegend=False
# ))

# Historical net line
fig.add_trace(go.Scatter(
    x=dfHistNet["Year"], y=dfHistNet["NetMt"],
    mode='lines', name='Net Emissions',
    line=dict(color='black', width=1)
))

fig.add_trace(go.Scatter(
    x=[2023, 2024],
    y=[666.3 + 1.5, 651.4 + 1.5],
    mode='markers',                          # only points
    # name='Net (Provisional)',            # shows in legend
    marker=dict(color='black', size=7, symbol='x'),
    showlegend=False
))


# Scenario lines + bands
add_scenario_lines(fig, dfOut, "Current-Policies-Med", "High-Ambition-Med", start_year=2025)
add_scenario_bands(fig, dfOut, "Current-Policies-Low", "Current-Policies-High", fillcolor='rgba(99,110,250,0.3)')
add_scenario_bands(fig, dfOut, "High-Ambition-Low", "High-Ambition-High", fillcolor='rgba(0,204,150,0.3)')

# Layout & grid
style_axes(fig, x_range=[1987, 2041], y_range=[-130, 830],
            x_ticks=list(range(1990, 2040, 5)),
            y_ticks=list(range(-100, 801, 100)))
# highlight 2025–2035
fig.add_vrect(x0=2025, x1=2035, fillcolor="LightBlue", opacity=0.1, layer="below", line_width=0)
# border
fig.add_shape(type="rect", xref="paper", yref="paper",
                x0=0, x1=1, y0=0, y1=1, line=dict(color="black", width=1), layer="above")
# grid
add_grid(fig, 1985, 2035, -100, 900)


fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='markers',
    marker=dict(size=0, color='rgba(0,0,0,0)'),
    name='<b><br><br>Policy Target</b>',          # bold legend title
    hoverinfo='none',
    showlegend=True
))

# 2030 NDC marker (optional: shown under same group)
fig.add_trace(go.Scatter(
    x=[2030], y=[783.8 * 0.6],
    mode='markers',
    name='2030 NDC',
    marker=dict(color='#750D86', size=8, symbol='triangle-up'),
    showlegend=True,
    legendgroup='NDC'
))

ndc_low_min, ndc_low_max = 289.5 + 34, 348.9 + 29.8
x_ndc = 2035
fig.add_trace(go.Scatter(
    x=[x_ndc, x_ndc],
    y=[ndc_low_min, ndc_low_max],
    mode='markers+lines',
    name='2035 NDC (w/o Offset)',
    line=dict(color="black", width=2, dash='dot'),
    marker=dict(symbol='triangle-up', size=8, color='black'),
    showlegend=True,
    legendgroup='NDC'
))

# 2030 NDC marker (optional: shown under same group)
# fig.add_trace(go.Scatter(
#     x=[2035], y=[289.5 + 34],
#     mode='markers',
#     name='Upper NDC Target<br>(W/O Offset)',
#     marker=dict(color='#750D86', size=8, symbol='triangle-up'),
#     showlegend=True,
#     legendgroup='NDC'
# ))


# Reduction annotations at 2035
x_center = 2038
fig.add_annotation(x=x_center, y=cp_med_2035,
                    text=f"<b>Current<br>Policies<br>-{red_cp:.1f}%</b>",
                    showarrow=False, font=dict(size=14, color="#636EFA"))
fig.add_annotation(x=x_center, y=ep_med_2035,
                    text=f"<b>High<br>Ambition<br>-{red_ep:.1f}%</b>",
                    showarrow=False, font=dict(size=14, color="#00CC96"))

fig.update_layout(
    xaxis_title=None
)


# pio.write_image(fig, FIG_TOTAL, width=1200, height=700, scale=2)
pio.write_image(fig, "./fig/total_emission.jpg", width=1200, height=700, scale=3)
pio.write_image(fig, "./fig/total_emission.svg", width=1200, height=700, scale=3)
fig

In [148]:
301.9898 - 289.5

12.489800000000002

In [147]:
1 - (348.9 + 29.8)/742.3

0.48982891014414653

In [146]:
1 - (289.5 + 34)/742.3

0.5641923750505187